### Response Completeness Evaluator

### Getting Started

This sample demonstrates how to use Response Completeness Evaluator
Before running the sample:
```bash
pip install azure-ai-projects azure-identity azure-ai-evaluation
```
Set these environment variables with your own values:
1) **PROJECT_CONNECTION_STRING** - The project connection string, as found in the overview page of your Azure AI Foundry project.
2) **MODEL_DEPLOYMENT_NAME** - The deployment name of the AI model, as found under the "Name" column in the "Models + endpoints" tab in your Azure AI Foundry project.
3) **AZURE_OPENAI_ENDPOINT** - Azure Open AI Endpoint to be used for evaluation.
4) **AZURE_OPENAI_API_KEY** - Azure Open AI Key to be used for evaluation.
5) **AZURE_OPENAI_API_VERSION** - Azure Open AI Api version to be used for evaluation.
6) **AZURE_SUBSCRIPTION_ID** - Azure Subscription Id of Azure AI Project
7) **PROJECT_NAME** - Azure AI Project Name
8) **RESOURCE_GROUP_NAME** - Azure AI Project Resource Group Name

The Response Completeness evaluator assesses the quality of an agent response by examining how well it aligns with the provided ground truth. The evaluation is based on the following scoring system:

<pre>
Score 1: Fully incomplete: The response misses all necessary and relevant information compared to the ground truth.
Score 2: Barely complete: The response contains only a small percentage of the necessary information.
Score 3: Moderately complete: The response includes about half of the necessary information.
Score 4: Mostly complete: The response contains most of the necessary information, with only minor omissions.
Score 5: Fully complete: The response perfectly matches all necessary and relevant information from the ground truth.
</pre>

The evaluation requires the following inputs:

Response: The response to be evaluated. (string)
Ground Truth: The correct and complete information against which the response is compared. (string)

The evaluator uses these inputs to determine the completeness score, ensuring that the response meaningfully addresses the query while adhering to the provided definitions and data.

### Initialize Completeness Evaluator


In [1]:
from azure.ai.evaluation import ResponseCompletenessEvaluator , AzureOpenAIModelConfiguration
from pprint import pprint
import os

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=os.environ["MODEL_DEPLOYMENT_NAME"],
)


In [2]:
from azure.ai.evaluation import ResponseCompletenessEvaluator , AzureOpenAIModelConfiguration
from pprint import pprint

# Set is_reasoning_model=True for models like gpt-5-nano that use max_completion_tokens
response_completeness_evaluator = ResponseCompletenessEvaluator(
    model_config=model_config,
    is_reasoning_model=True
)

Class ResponseCompletenessEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


### Samples

#### Evaluating for a ground_truth and prediction

In [3]:
result = response_completeness_evaluator(
    response="The capital of Japan",
    ground_truth="The capital of Japan is Tokyo."
)
result

{'response_completeness': 1,
 'response_completeness_result': 'fail',
 'response_completeness_threshold': 3,
 'response_completeness_reason': 'The response "The capital of Japan" omits the essential information "is Tokyo" from the ground truth, so it does not fully reflect the claim. It contains only a partial phrase, not the complete statement. Therefore it is fully incomplete (score 1).',
 'response_completeness_prompt_tokens': 1353,
 'response_completeness_completion_tokens': 1188,
 'response_completeness_total_tokens': 2541,
 'response_completeness_finish_reason': 'stop',
 'response_completeness_model': 'gpt-5-nano-2025-08-07',
 'response_completeness_sample_input': '[{"role": "user", "content": "{\\"response\\": \\"The capital of Japan\\", \\"ground_truth\\": \\"The capital of Japan is Tokyo.\\"}"}]',
 'response_completeness_sample_output': '[{"role": "assistant", "content": "<S0>I\'m unable to share step-by-step chain-of-thought, but here\'s a concise assessment.</S0>\\n<S1>The r

In [4]:
result = response_completeness_evaluator(
    response="The capital of Japan is Tokyo.",
    ground_truth="The capital of Japan is Tokyo."
)
result

{'response_completeness': 5,
 'response_completeness_result': 'pass',
 'response_completeness_threshold': 3,
 'response_completeness_reason': 'The response is a perfect match to the ground truth, containing all and only the information present there.',
 'response_completeness_prompt_tokens': 1355,
 'response_completeness_completion_tokens': 658,
 'response_completeness_total_tokens': 2013,
 'response_completeness_finish_reason': 'stop',
 'response_completeness_model': 'gpt-5-nano-2025-08-07',
 'response_completeness_sample_input': '[{"role": "user", "content": "{\\"response\\": \\"The capital of Japan is Tokyo.\\", \\"ground_truth\\": \\"The capital of Japan is Tokyo.\\"}"}]',
 'response_completeness_sample_output': '[{"role": "assistant", "content": "<S0> I\'m unable to share step-by-step chain-of-thought, but here is a concise assessment: The given response exactly matches the ground truth. </S0>\\n<S1> The response is a perfect match to the ground truth, containing all and only the 

#### Evaluate with a reasoning model

In [5]:
from azure.ai.evaluation import ResponseCompletenessEvaluator , AzureOpenAIModelConfiguration
from pprint import pprint

# set is_reasoning_model to True in case the model is a reasoning model (ex: o3-mini, o1-preview)
response_completeness_evaluator = ResponseCompletenessEvaluator(model_config=model_config,
                                                                is_reasoning_model=True)

result = response_completeness_evaluator(
    response="The capital of Japan is Tokyo.",
    ground_truth="The capital of Japan is Tokyo."
)
result

{'response_completeness': 5,
 'response_completeness_result': 'pass',
 'response_completeness_threshold': 3,
 'response_completeness_reason': 'The response reproduces the ground truth sentence exactly; thus it is fully complete (score 5).',
 'response_completeness_prompt_tokens': 1355,
 'response_completeness_completion_tokens': 527,
 'response_completeness_total_tokens': 1882,
 'response_completeness_finish_reason': 'stop',
 'response_completeness_model': 'gpt-5-nano-2025-08-07',
 'response_completeness_sample_input': '[{"role": "user", "content": "{\\"response\\": \\"The capital of Japan is Tokyo.\\", \\"ground_truth\\": \\"The capital of Japan is Tokyo.\\"}"}]',
 'response_completeness_sample_output': '[{"role": "assistant", "content": "<S0>I can\\u2019t share step-by-step thoughts, but here is a concise answer: The response exactly matches the ground truth, so it is fully complete.</S0>\\n<S1>The response reproduces the ground truth sentence exactly; thus it is fully complete (scor

# Batch run for response completeness

In [6]:
import json
import pandas as pd

data = [
    {
        "response": "The temperature of Seattle now is 70 degrees. Based on the temperature, having an outdoor office party is recommended.",
        "ground_truth": "The temperature of Seattle now is 50 degrees. It will be recommended to bring a jacket in the evening.",
    },
    {
        "response": "The email draft \"Project Plan\" is attached. Please review and provide feedback.",
        "ground_truth": "The email draft \"Project Plan\" is attached. Please review and provide feedback by EOD.",
    },
    {
        "response": "Based on the retrieved documents, the shareholder meeting discussed the operational efficiency of the company and financing options.",
        "ground_truth": "The shareholder meeting discussed the compensation package of the company CEO.",
    }
]

file_path = "response_completeness_data.jsonl"

pd.DataFrame(data).to_json(
    file_path, orient="records", lines=True
)

from azure.ai.evaluation import evaluate

# azure_ai_project={
#         "subscription_id": os.environ["AZURE_SUBSCRIPTION_ID"],
#         "project_name": os.environ["PROJECT_NAME"],
#         "resource_group_name": os.environ["RESOURCE_GROUP_NAME"],
#     }
azure_ai_project=os.environ["PROJECT_ENDPOINT"]

response = evaluate(
    data=file_path,
    evaluators={
        "response_completeness": response_completeness_evaluator,
    },
    azure_ai_project=azure_ai_project,
)

pprint(f'AI Foundry URL: {response.get("studio_url")}')

2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Finished 1 / 3 lines.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Average execution time for completed lines: 5.03 seconds. Estimated time for incomplete lines: 10.06 seconds.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Average execution time for completed lines: 5.03 seconds. Estimated time for incomplete lines: 10.06 seconds.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Finished 2 / 3 lines.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Average execution time for completed lines: 2.93 seconds. Estimated time for incomplete lines: 2.93 seconds.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Finished 2 / 3 lines.
2025-11-10 14:03:48 +0100 6306689024 execution.bulk     INFO     Average execution time for completed lines: 2.93 seconds. Estimated time for incomplete lines: 2.93 seconds.
2025-11-10 14:03:51 +0100 6306689024 exec

Aggregated metrics for evaluator is not a dictionary will not be logged as metrics


======= Run Summary =======

Run name: "response_completeness_20251110_130342_980387"
Run status: "Completed"
Start time: "2025-11-10 13:03:42.980387+00:00"
Duration: "0:00:09.010297"

======= Combined Run Summary (Per Evaluator) =======

{
    "response_completeness": {
        "status": "Completed",
        "duration": "0:00:09.010297",
        "completed_lines": 3,
        "failed_lines": 0,
        "log_path": null,
        "error_message": null,
        "error_code": null
    }
}


'AI Foundry URL: None'
'AI Foundry URL: None'


[{"variableName": "data", "type": "list", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.list"}, {"variableName": "model_config", "type": "dictionary", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.dict"}, {"variableName": "response", "type": "dictionary", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.dict"}, {"variableName": "result", "type": "dictionary", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.dict"}]
[{"variableName": "data", "type": "list", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.list"}, {"variableName": "model_config", "type": "dictionary", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.dict"}, {"variableName": "response", "type": "dictionary", "supportedEngines": ["pandas"], "isLocalVariable": false, "rawType": "builtins.dict"}, {"variableName": "result", "type": "dictionary